In [5]:
import pandas as pd
from pathlib import Path

parquet_path = Path("../../data/raw/NFI/nfi_particle_data_full.parquet")
nfi_df = pd.read_parquet(parquet_path)
nfi_df.shape

(2801667, 106)

In [15]:
# Confirm stub_id + particle_id uniqueness
nfi_df.duplicated(subset=["stub_id", "particle_id"]).sum()

np.int64(0)

In [6]:
# Subset NFI data to only include id + element columns
element_cols = list(nfi_df.loc[:, 'ac':'zr'].columns)
elements_df = nfi_df[['stub_id','particle_id'] + element_cols]
elements_df.columns

Index(['stub_id', 'particle_id', 'ac', 'ag', 'al', 'ar', 'as', 'at', 'au', 'b',
       'ba', 'bi', 'br', 'ca', 'cd', 'ce', 'cl', 'co', 'cr', 'cs', 'cu', 'dy',
       'er', 'eu', 'f', 'fe', 'fr', 'ga', 'gd', 'ge', 'hf', 'hg', 'ho', 'i',
       'in', 'ir', 'k', 'kr', 'la', 'lu', 'mg', 'mn', 'mo', 'n', 'na', 'nb',
       'nd', 'ne', 'ni', 'np', 'o', 'os', 'p', 'pa', 'pb', 'pd', 'pm', 'po',
       'pr', 'pt', 'pu', 'ra', 'rb', 're', 'rh', 'rn', 'ru', 's', 'sb', 'sc',
       'se', 'si', 'sm', 'sn', 'sr', 'ta', 'tb', 'tc', 'te', 'th', 'ti', 'tl',
       'tm', 'u', 'v', 'w', 'xe', 'y', 'yb', 'zn', 'zr'],
      dtype='str')

In [7]:
# Use non-zero rates from particle_eda.ipynb to filter out non-informative elements
nonzero_rates = (elements_df[element_cols] > 0).mean().sort_values(ascending=False)
informative_cols = sorted(nonzero_rates[nonzero_rates > 0.01].index.tolist())
informative_df = elements_df[['stub_id', 'particle_id'] + informative_cols]
informative_df.columns

Index(['stub_id', 'particle_id', 'al', 'as', 'ba', 'br', 'ca', 'cl', 'cr',
       'cu', 'fe', 'hg', 'k', 'mg', 'mn', 'mo', 'na', 'ni', 'o', 'p', 'pb',
       's', 'sb', 'si', 'sn', 'sr', 'ti', 'w', 'zn'],
      dtype='str')

In [8]:
# Confirm number of informative features/elements kept
informative_df.shape

(2801667, 29)

In [9]:
# Confirm that NFI elemental composition includes oxygen in the percentage (in contrast to NIST)
informative_df['sum_with_oxygen'] = informative_df.loc[:, 'al':'zn'].sum(axis=1).round(2)
informative_df['sum_without_oxygen'] = informative_df.loc[:, 'al':'zn'].drop('o', axis=1).sum(axis=1).round(2)
informative_df[['sum_with_oxygen', 'sum_without_oxygen']].head()

,sum_with_oxygen,sum_without_oxygen
0,100.0,77.00
1,100.0,55.35
2,100.0,56.88
3,100.0,61.73
4,100.0,58.39


In [10]:
# Add the GSR / Non-GSR labels from particle_labeled.parquet
p_df = pd.read_parquet("../../data/processed/particle_labeled.parquet")
p_labeled = p_df[['stub_id', 'particle_id', 'label']]
labeled_df = informative_df.copy()
labeled_df = labeled_df.drop(['sum_with_oxygen', 'sum_without_oxygen'], axis=1)
labeled_df = labeled_df.merge(p_labeled, on=['stub_id', 'particle_id'])
labeled_df.shape

(2801667, 30)

In [11]:
# Confirm Number of GSR vs Non-GSR particles
labeled_df['label'].value_counts()

label
Non_GSR      1216039
GSR          1078946
Ambiguous     506682
Name: count, dtype: int64

In [12]:
# Drop ambiguous particles
bin_labeled_df = labeled_df[labeled_df['label'] != 'Ambiguous']
assert bin_labeled_df.shape[0] == labeled_df.shape[0] - labeled_df['label'].value_counts()['Ambiguous']

In [13]:
# New shape
bin_labeled_df.shape

(2294985, 30)

In [14]:
# Write to parquet
bin_labeled_df.to_parquet("../../data/processed/nfi_relevant_elements_labeled.parquet")